In [ ]:
# Default parameters for manual testing. 
# ADF will automatically overwrite these values at runtime.

file_list = "suppliers.csv"
new_status = "Success"

In [ ]:
# ========== UPDATE LOGIC ==========
print(f"Updating files: {file_list} to status: {new_status}")

files = [f.strip() for f in file_list.split(",") if f.strip()]

control_path = "abfss://medallion@stlakesupply.dfs.core.windows.net/control/ingestion_status.csv"

import pandas as pd
from datetime import datetime

df = pd.read_csv(control_path)
df['ingestion_time'] = df['ingestion_time'].astype(str)

updated = 0
for file in files:
    if file in df['source_file_name'].values:
        df.loc[df['source_file_name'] == file, 'ingestion_status'] = new_status
        df.loc[df['source_file_name'] == file, 'ingestion_time'] = datetime.utcnow().isoformat()
        updated += 1
    else:
        print(f"Warning: {file} not found in control table")

if updated > 0:
    df.to_csv(control_path, index=False)
    print(f"Updated {updated} files to {new_status}")
else:
    print("No files updated")